In [2]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = os.environ.get("BIRDSEYE_DATA", "/home/keyaan/Project/data/SPY/0-dte/train")

In [6]:
df = pd.read_parquet(f"{DATA_DIR}/20240208.parquet")
display(df)

,open,high,low,close,volume,count,spot,460.00_CE_lastQuoteTimestamp,460.00_CE_lastTradeTimestamp,460.00_ce_bid_0,...,535.00_PE_totalTradeQty,535.00_pe_ttv,535.00_pe_ttq,535.00_pe_vwap,535.00_PE_open,535.00_PE_high,535.00_PE_low,535.00_PE_close,535.00_pe_premium,atm_strike
timestamp,,,,,,,,,,,,,,,,,,,,,
2024-02-08 10:30:00-05:00,498.1300,498.1700,498.11,498.16,21536.0,661.0,498.16,1.707406e+18,NaN,37.77,...,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,36.880,498.0
2024-02-08 10:30:01-05:00,498.1300,498.1700,498.11,498.16,21536.0,661.0,498.16,1.707406e+18,NaN,37.77,...,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,36.860,498.0
2024-02-08 10:30:02-05:00,498.1600,498.1600,498.10,498.12,5718.0,154.0,498.12,1.707406e+18,NaN,37.76,...,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,36.835,498.0
2024-02-08 10:30:03-05:00,498.1200,498.1600,498.08,498.09,12956.0,172.0,498.09,1.707406e+18,NaN,37.70,...,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,36.860,498.0
2024-02-08 10:30:04-05:00,498.0981,498.0981,498.04,498.05,4480.0,143.0,498.05,1.707406e+18,NaN,37.75,...,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,36.905,498.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-02-08 15:59:56-05:00,0.0000,0.0000,0.00,498.26,100.0,3.0,498.26,1.707426e+18,NaN,38.03,...,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,36.745,498.0
2024-02-08 15:59:57-05:00,498.2600,498.2600,498.25,498.25,656.0,9.0,498.25,1.707426e+18,NaN,38.03,...,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,36.745,498.0
2024-02-08 15:59:58-05:00,0.0000,0.0000,0.00,498.25,0.0,0.0,498.25,1.707426e+18,NaN,38.03,...,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,36.745,498.0


In [ ]:
from engine import BirdsEye
from strategies.range_short_strangle import RangeShortStrangle

if __name__ == "__main__":
    be = BirdsEye(
        data_dir          = "/home/keyaan/Project/data/SPY/0-dte/train",
        strategy_cls      = RangeShortStrangle,
        index             = "SPY",
        lot_size          = 100,
        starting_cash     = 1_000_000.0,
        margin_per_lot    = 10000.0,
        strategy_kwargs   = {"lots": 10, "max_lots": 10},
        cost_kwargs       = {"txn_cost_per_lot": 0.85},
        n_workers         = 20,
        collect_perseclog = True,
    )
    
    res = be.run()

In [2]:
import matplotlib.pyplot as plt

DAY = "20240208"

# ---- 1. per-day summary table ----
print("=== per-day summary ===")
display(res.summary)


=== per-day summary ===


,fills,gross($),costs($),net($)
day,,,,
20240102,132,-260.0,324.0,-584.0
20240104,136,103.0,378.0,-275.0
20240105,110,-490.5,270.5,-761.0
20240109,136,52.0,378.0,-326.0
20240110,136,-31.5,379.5,-411.0
...,...,...,...,...
20250804,136,383.0,378.0,5.0
20250806,136,239.0,380.0,-141.0
20250807,44,-635.0,112.0,-747.0


In [6]:
def display_stats(stats: dict):
    from IPython.display import display, HTML

    def fmt(k, v):
        if isinstance(v, float):
            if "pct" in k or "cagr" in k or "calmar" in k:
                color = "green" if v > 0 else "red"
                return f'<span style="color:{color}">{v:+.2f}%</span>'
            if "pnl" in k or "day" in k or "win" in k or "loss" in k or "cost" in k:
                color = "green" if v > 0 else ("red" if v < 0 else "inherit")
                return f'<span style="color:{color}">${v:+,.2f}</span>'
            return f"{v:.4f}"
        if isinstance(v, int):
            return f"{v:,}"
        return str(v)

    rows = "".join(
        f"<tr><td style='padding:4px 16px 4px 0;color:gray;font-size:13px'>{k}</td>"
        f"<td style='padding:4px 0;font-size:13px;font-weight:500'>{fmt(k,v)}</td></tr>"
        for k, v in stats.items()
    )
    display(HTML(f"<table style='border-collapse:collapse'>{rows}</table>"))

display_stats(res.stats())

n_days,277
total_pnl,"$-51,421.00"
avg_day,$-185.64
pct_pos_days,+0.24%
pct_neg_days,+0.62%
avg_win,$+116.36
avg_loss,$-342.29
best_day,$+455.00
worst_day,"$-1,714.00"
cagr_gross,+10.13%
cagr_net,-23.39%


In [ ]:

# ---- 5. trade ledger (all days) ----
led = res.tradelog
print(f"=== trade ledger: {len(led)} fills ===")
display(led.head(10))                                  # alpha_* cols show fire-time values


=== trade ledger: 26806 fills ===


,day,timestamp,strike,opt_type,action,lots,fill_price,txn_cost,brokerage,spread_cost,...,signal,note,alpha_sec,alpha_spot,alpha_range_bps,alpha_ce_strike,alpha_pe_strike,alpha_ce_mid,alpha_pe_mid,alpha_current_prem
0,20240102,12:01:16,473.0,CE,SELL,10.0,0.28,8.50,0.0,5.0,...,calm_entry,,5475.0,471.88,9.96,473.0,471.0,0.28,0.34,NaN
1,20240102,12:01:16,471.0,PE,SELL,10.0,0.34,8.50,0.0,5.0,...,calm_entry,,5475.0,471.88,9.96,473.0,471.0,0.28,0.34,NaN
2,20240102,12:31:16,473.0,CE,BUY,1.0,0.38,0.85,0.0,0.5,...,hold_elapsed,square off,7275.0,472.37,10.69,474.0,471.0,0.15,0.23,0.6
3,20240102,12:31:16,471.0,PE,BUY,1.0,0.22,0.85,0.0,0.5,...,hold_elapsed,square off,7275.0,472.37,10.69,474.0,471.0,0.15,0.23,0.6
4,20240102,12:31:17,473.0,CE,BUY,1.0,0.38,0.85,0.0,0.5,...,hold_elapsed,square off,7275.0,472.37,10.69,474.0,471.0,0.15,0.23,0.6
5,20240102,12:31:17,471.0,PE,BUY,1.0,0.22,0.85,0.0,0.5,...,hold_elapsed,square off,7275.0,472.37,10.69,474.0,471.0,0.15,0.23,0.6
6,20240102,12:31:18,473.0,CE,BUY,1.0,0.38,0.85,0.0,0.5,...,hold_elapsed,square off,7275.0,472.37,10.69,474.0,471.0,0.15,0.23,0.6
7,20240102,12:31:18,471.0,PE,BUY,1.0,0.22,0.85,0.0,0.5,...,hold_elapsed,square off,7275.0,472.37,10.69,474.0,471.0,0.15,0.23,0.6
8,20240102,12:31:19,473.0,CE,BUY,1.0,0.38,0.85,0.0,0.5,...,hold_elapsed,square off,7275.0,472.37,10.69,474.0,471.0,0.15,0.23,0.6
9,20240102,12:31:19,471.0,PE,BUY,1.0,0.22,0.85,0.0,0.5,...,hold_elapsed,square off,7275.0,472.37,10.69,474.0,471.0,0.15,0.23,0.6


In [7]:

# ---- 6. per-second log for one day ----
sl = res.perseclog(DAY)
print(f"=== per-second log {DAY}: {len(sl)} rows ===")
display(sl.iloc[2000:2005])                            # spot, atm, state, every alpha

=== per-second log 20240208: 19723 rows ===


,timestamp,spot,atm,state,sec,range_bps,ce_strike,pe_strike,ce_mid,pe_mid,current_prem
2000,1707408200000000000,498.16,498.0,WAIT,2000.0,9.64,499.0,497.0,0.52,0.42,NaN
2001,1707408201000000000,498.16,498.0,WAIT,2001.0,9.64,499.0,497.0,0.52,0.42,NaN
2002,1707408202000000000,498.17,498.0,WAIT,2002.0,9.64,499.0,497.0,0.52,0.42,NaN
2003,1707408203000000000,498.17,498.0,WAIT,2003.0,9.64,499.0,497.0,0.52,0.42,NaN
2004,1707408204000000000,498.17,498.0,WAIT,2004.0,9.64,499.0,497.0,0.52,0.42,NaN
